# Pipeline: sinistros de trânsito em Fortaleza (2015-2024)

Este notebook executa o projeto inteiro de ponta a ponta com a classe `TrafegoPipeline`
(`src/sinistros/pipeline.py`). É o mesmo pipeline apresentado antes, com a mesma classe e os
mesmos seis métodos, agora com a limpeza incluída e as análises corrigidas.

![Pipeline](../figuras/00_pipeline.png)

| Fase do diagrama | Método da classe | Notebook com o detalhe |
|---|---|---|
| 1. Ingestão | `etapa_1_ingestao` | `01_limpeza` |
| 2. Transformação e enriquecimento | `etapa_1_ingestao` (pré-processamento) | `01_limpeza` |
| 3. Modelo analítico e visualização | `etapa_2_analise_descritiva`, `etapa_3_analise_espacial`, `etapa_4_veiculos_coocorrencia` | `02_exploratoria`, `04_vias_mapa` |
| 4. Validação de hipóteses | `etapa_5_testes_inferenciais`, `etapa_6_verificacao_hipoteses` | `03_inferencia` |

Saídas: base tratada em `dados/processados/`, gráficos em `figuras/` e números-chave em
`resultados/resultados.json` (usados para gerar os relatórios em PDF).

In [1]:
import sys
import warnings
from pathlib import Path

# Encontra a raiz do projeto (a pasta que contém src/sinistros) e importa o pacote
raiz = Path.cwd()
while not (raiz / "src" / "sinistros").exists():
    raiz = raiz.parent
sys.path.insert(0, str(raiz / "src"))
warnings.filterwarnings("ignore")

import pandas as pd
from IPython.display import Image, display

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

from sinistros import config, limpeza, analises as an, graficos as gr

## Execução

In [2]:
from sinistros.pipeline import TrafegoPipeline

pipeline = TrafegoPipeline()
pipeline.executar();

>>> Etapa 1: Ingestão e pré-processamento


Base bruta: 130,820 registros e 27 colunas
Dados externos: malha e população dos bairros (IBGE 2022), malha viária (OpenStreetMap)


- Leitura: 130,820 linhas afetadas
- Duplicatas: 54 linhas afetadas
- Tempo: 0 linhas afetadas
- Bairro: 354 linhas afetadas
- Vias: 5,275 linhas afetadas
- Categorias: 17,425 linhas afetadas
- Categorias: 79,764 linhas afetadas
- Categorias: 28,510 linhas afetadas
- Categorias: 1,254 linhas afetadas
- Severidade: 11,601 linhas afetadas
- Veículos: 11,969 linhas afetadas
Base tratada: 130,766 registros, 39 colunas -> dados\processados\sinistros_tratados.parquet

>>> Etapa 2: Análise descritiva e temporal (passos 1 a 3)


130,766 acidentes, 1,986 mortos; letalidade 1.47%
Pico às 16h; letalidade na madrugada 2.15% vs. manhã 1.04%



>>> Etapa 3: Análise espacial (bairros e vias)
Top 3 bairros em acidentes: Centro (7,257), Aldeota (5,404), Meireles (4,127)
Top 3 vias: Av. Washington Soares / CE-040 (3,539), BR-116 (3,315), Avenida General Osório de Paiva (2,669)


...


>>> Etapa 5: Testes inferenciais (passos 9 e 10)


Controle x severidade: V = 0.049 (era 0.237 com 'não informado')
Cruzamento x severidade: V = 0.127



>>> Etapa 6: Validação das 5 hipóteses (passo 11)


H1: OR madrugada 1.90, fim de semana 1.58
H2: com moto, 96.5% têm vítima (sem moto: 60.3%)
H3: quebra em 2018-01; mortes -6.6% ao ano
H4: sem controle vs. semáforo, OR 1.62
H5: pseudo-R² 0.39 (versão anterior circular: 0.94)



Resultados salvos em resultados\resultados.json
Gráficos salvos em figuras/

Pipeline executado de ponta a ponta.


## Registro da limpeza

In [3]:
pipeline.log_limpeza

,etapa,linhas_afetadas,linhas_resultantes,descricao
0,Leitura,130820,130820,Base recebida da etapa de ETL anterior (3 regi...
1,Duplicatas,54,130766,Linhas idênticas em todas as 27 colunas removi...
2,Tempo,0,130766,"Dia da semana, fim de semana e faixa horária r..."
3,Bairro,354,130766,Bairro reatribuído pelas coordenadas com a mal...
4,Vias,5275,130766,Nomes de via padronizados: 11.502 grafias -> 6...
5,Categorias,17425,130766,'tipo_cruzamento': rótulos padronizados; 'não ...
6,Categorias,79764,130766,'controle_trafego': rótulos padronizados; 'não...
7,Categorias,28510,130766,'uso_solo': rótulos padronizados; 'não informa...
8,Categorias,1254,130766,'natureza': rótulos padronizados; 'não informa...
9,Severidade,11601,130766,Severidade recalculada pelas contagens de víti...


## Resumo das 5 hipóteses

In [4]:
r = pipeline.resultados
resumo = pd.DataFrame([
    ["H1", "Madrugada e fim de semana aumentam a letalidade",
     f"OR madrugada {r['h1']['or_ajustado']['Madrugada (vs. manhã)']['or']:.2f}; "
     f"OR fim de semana {r['h1']['or_ajustado']['Fim de semana (vs. dia útil)']['or']:.2f}",
     "Confirmada (efeitos independentes)"],
    ["H2", "Motos aumentam o número de vítimas",
     f"{r['h2']['pct_com_vitima_moto']:.1f}% dos acidentes com moto têm vítima "
     f"(sem moto: {r['h2']['pct_com_vitima_sem_moto']:.1f}%); "
     f"vítimas {100 * (r['h2']['irr']['Só moto']['irr'] - 1):.0f}% maiores",
     "Confirmada"],
    ["H3", "Os acidentes caem ao longo do tempo",
     f"Degrau em {r['h3']['quebra_mes']} ({r['h3']['degrau_mensal']:.0f}/mês), sem tendência "
     f"contínua; mortes {r['h3']['mortes_variacao_anual_pct']:.1f}% ao ano",
     "Revista: patamar, não tendência; mortes caem"],
    ["H4", "O controle de tráfego afeta a severidade",
     f"V = {r['h4']['v_cramer_sem_ni']:.2f} (era {r['h4']['v_cramer_com_ni']:.2f}); "
     f"sem controle vs. semáforo OR {r['h4']['or']['Sem controle (vs. semáforo)']['or']:.2f}",
     "Parcial: efeito fraco, só em 39% da base"],
    ["H5", "Exposição explica os acidentes por bairro",
     f"Pseudo-R² {r['h5']['pseudo_r2']['População + km de vias']:.2f} "
     f"(antes: R² {r['h5']['r2_circular_versao_anterior']:.2f}, circular)",
     "Revista: explica ~40%; Centro e corredores comerciais em excesso"],
], columns=["Hipótese", "Enunciado", "Resultado", "Conclusão"])
resumo

,Hipótese,Enunciado,Resultado,Conclusão
0,H1,Madrugada e fim de semana aumentam a letalidade,OR madrugada 1.90; OR fim de semana 1.58,Confirmada (efeitos independentes)
1,H2,Motos aumentam o número de vítimas,96.5% dos acidentes com moto têm vítima (sem m...,Confirmada
2,H3,Os acidentes caem ao longo do tempo,"Degrau em 2018-01 (-585/mês), sem tendência co...","Revista: patamar, não tendência; mortes caem"
3,H4,O controle de tráfego afeta a severidade,V = 0.05 (era 0.24); sem controle vs. semáforo...,"Parcial: efeito fraco, só em 39% da base"
4,H5,Exposição explica os acidentes por bairro,"Pseudo-R² 0.39 (antes: R² 0.94, circular)",Revista: explica ~40%; Centro e corredores com...


## Gráficos gerados

In [5]:
for arquivo in sorted(config.FIGURAS.glob("*.png")):
    print(arquivo.name)
for arquivo in sorted(config.MAPAS.glob("*")):
    print("mapas/" + arquivo.name)

00_pipeline.png
01_qualidade_nao_informado_por_ano.png
02_acidentes_e_mortes_por_ano.png
03_severidade.png
04_acidentes_por_hora.png
05_mapa_calor_dia_hora.png
06_letalidade_faixa_fim_de_semana.png
07_bairros_total_e_taxa.png
08_mapa_bairros_acidentes_por_habitante.png
09_mapa_bairros_mortes_por_habitante.png
10_bairros_mortes_por_habitante_ic.png
11_veiculos_envolvidos.png
12_coocorrencia_veiculos.png
13_letalidade_por_veiculo.png
14_letalidade_por_natureza.png
15_correlacao_spearman.png
16_qui_quadrado_residuos.png
17_vitimas_por_natureza.png
18_h1_razao_de_chances.png
19_h2_motos_vitimas.png
20_h3_tendencia.png
21_h4_controle_trafego.png
22_h5_bairros_observado_esperado.png
23_vias_top20.png
mapas/24_mapa_vias_top8.png
mapas/mapa_interativo.html
